# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [1]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime # nodig voor dimensies (begin/eindtijd)

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

2026-03-29 22:00:55.284 | INFO     | __main__:<module>:11 - SDM en DWH connecties succesvol.


Dim_Klant (SCD TYPE 2) ROBBERT

Dim_Accessoire (SCD TYPE 1) ROBBERT

Dim_Datum ROBBERT

Fct_Verkoop ROBBERT


Dim_Fiets (SCD Type 2) MEES

In [ ]:
logger.info("Ophalen Fiets uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, naam, adres, plaats
FROM (
    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Inkoop_Fiets f
    JOIN Fiets_Inkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Verkoop_Fiets f
    JOIN Fiets_Verkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Onderhoud_Fiets f
    JOIN Onderhoud_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
)
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
    fiets_sk
FROM Dim_Fiets
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

for row in source_data:
    (fietsnr, soort, merk, type_, kleur,
     fabrikantnr, fab_naam, fab_adres, fab_plaats) = row

    now = datetime.now()

    if fietsnr not in dwh_data:
        # nieuw
        logger.info(f"Nieuwe fiets: {fietsnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Fiets (
            fietsnr, soort, merk, type, kleur,
            fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (fietsnr, soort, merk, type_, kleur,
              fabrikantnr, fab_naam, fab_adres, fab_plaats,
              now))

    else:
        # check wijzigingen
        dwh_row = dwh_data[fietsnr]

        (_, d_soort, d_merk, d_type, d_kleur,
         d_fabnr, d_fabnaam, d_fabadres, d_fabplaats,
         fiets_sk) = dwh_row

        if (soort, merk, type_, kleur,
            fabrikantnr, fab_naam, fab_adres, fab_plaats) != \
           (d_soort, d_merk, d_type, d_kleur,
            d_fabnr, d_fabnaam, d_fabadres, d_fabplaats):

            logger.info(f"Update in fiets: {fietsnr}")

            # 1. Sluit oude (voeg eindtijd toe)
            dwh_cursor.execute("""
            UPDATE Dim_Fiets
            SET eindtijd = ?
            WHERE fiets_sk = ?
            """, (now, fiets_sk))

            # 2. Maak nieuwe (met nieuwe gegevens en begintijd)
            dwh_cursor.execute("""
            INSERT INTO Dim_Fiets (
                fietsnr, soort, merk, type, kleur,
                fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
                begintijd, eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
            """, (fietsnr, soort, merk, type_, kleur,
                  fabrikantnr, fab_naam, fab_adres, fab_plaats,
                  now))

dwh_conn.commit()
logger.info("Fiets dimensie geupdate in DWH.")

2026-03-29 22:01:33.322 | INFO     | __main__:<module>:1 - Ophalen Fiets uit SDM
2026-03-29 22:01:33.326 | INFO     | __main__:<module>:44 - Data opgehaald uit SDM en DWH.
2026-03-29 22:01:33.327 | INFO     | __main__:<module>:80 - Wijziging fiets: 1
C:\Users\MeesR\AppData\Local\Temp\ipykernel_25556\998742227.py:83: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_cursor.execute("""
C:\Users\MeesR\AppData\Local\Temp\ipykernel_25556\998742227.py:90: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_cursor.execute("""
2026-03-29 22:01:33.333 | INFO     | __main__:<module>:80 - Wijziging fiets: 3
2026-03-29 22:01:33.333 | INFO     | __main__:<module>:80 - Wijziging fiets: 4
2026-03-29 22:01:33.335 | INFO     | __main__:<module>:80 - Wijziging fiets: 11
2026-03-29 22:01:33.336 | INFO     

Dim_Monteur (SCD Type 1) MEES

In [ ]:
logger.info("Ophalen Monteur uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Accessoire_Verkoop_Monteur m
JOIN Accessoire_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL
                                   
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Fiets_Verkoop_Monteur m
JOIN Fiets_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL

SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Onderhoud_Monteur m
JOIN Onderhoud_Filiaal f ON m.filiaal = f.filiaalnr
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    monteurnr, naam, woonplaats, uurloon,
    filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
    monteur_sk
FROM Dim_Monteur
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

processed_keys = set()

for row in source_data:
    (monteurnr, naam, woonplaats, uurloon,
     filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) = row

    if monteurnr in processed_keys:
        continue
    processed_keys.add(monteurnr)

    now = datetime.now()

    if monteurnr not in dwh_data:
        # nieuwe regels worden toegevoegd
        logger.info(f"Nieuwe monteur: {monteurnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Monteur (
            monteurnr, naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (monteurnr, naam, woonplaats, uurloon,
              filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
              now))

    else:
        # bestaande regels worden overschreven als er wijzigingen zijn
        (_, d_naam, d_woonplaats, d_uurloon,
         d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie,
         monteur_sk) = dwh_data[monteurnr]

        if (naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) != \
           (d_naam, d_woonplaats, d_uurloon,
            d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie):

            logger.info(f"Update monteur (Type 1): {monteurnr}")

            dwh_cursor.execute("""
            UPDATE Dim_Monteur
            SET naam = ?, woonplaats = ?, uurloon = ?,
                filiaalnr = ?, filiaalnaam = ?, filiaaladres = ?, filiaalprovincie = ?
            WHERE monteur_sk = ?
            """, (naam, woonplaats, uurloon,
                  filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
                  monteur_sk))
            
dwh_conn.commit()
logger.info("Monteur dimensie geupdate in DWH.")

2026-03-29 22:08:03.006 | INFO     | __main__:<module>:1 - Ophalen Monteur uit SDM
2026-03-29 22:08:03.009 | INFO     | __main__:<module>:108 - Monteur dimensie geupdate in DWH.


Fct_Onderhoud MEES

Fct_Inkoop MEES